# Day 3 · Lab 1 — SaaS MCP Wrapper with OAuth2

## What you'll build

1. A **mock OAuth2 token endpoint** (in-process httpx server)
2. A **TokenCache** class — refresh 5 minutes before expiry
3. A **FastMCP-style tool** that:
   - Fetches token from cache
   - Calls a simulated SaaS endpoint with Bearer auth
   - Handles 401 → refresh → retry once
4. An **agent-side call pattern** that never sees auth complexity

## Sandbox requirements

- `~/agentic-lab/.env` with `OPENROUTER_API_KEY`, `DATABASE_URL`
- `httpx` package (installed with `pip install --user httpx tenacity`)
- Same kernel: `/opt/miniconda/bin/python` (base)

## Step 1 — Environment

In [ ]:
import os, sys, subprocess
from pathlib import Path

for pkg in ["python-dotenv", "httpx", "tenacity"]:
    try:
        __import__(pkg.replace("-", "_").split("[")[0])
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

from dotenv import load_dotenv
load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "":
        del os.environ[k]

print("✓ Environment ready")
print(f"  OPENROUTER_API_KEY: {'set' if os.environ.get('OPENROUTER_API_KEY') else 'MISSING'}")

## Step 2 — Mock OAuth2 server (in-notebook)

For the lab, we simulate an OAuth2 auth server and a SaaS resource server as async functions. In production these are separate services.

In [ ]:
import time
import uuid


class MockAuthServer:
    """Simulates a token endpoint."""

    VALID_CLIENT_ID = "day3-lab-client"
    VALID_SECRET = "day3-lab-secret"
    TOKEN_LIFETIME = 60   # 60 seconds — short so we can watch cache behavior

    def __init__(self):
        self.issued = {}
        self.fetch_count = 0

    async def token_endpoint(self, client_id: str, client_secret: str) -> dict:
        self.fetch_count += 1
        if client_id != self.VALID_CLIENT_ID or client_secret != self.VALID_SECRET:
            raise PermissionError("Invalid client credentials")
        token = f"tok-{uuid.uuid4().hex[:12]}"
        self.issued[token] = time.time() + self.TOKEN_LIFETIME
        return {"access_token": token, "expires_in": self.TOKEN_LIFETIME, "token_type": "Bearer"}

    async def validate(self, token: str) -> bool:
        expires = self.issued.get(token)
        return expires is not None and expires > time.time()


auth_server = MockAuthServer()
print("✓ Mock OAuth2 auth server ready")

## Step 3 — Mock SaaS resource server

In [ ]:
class MockSaaS:
    """Simulates a Salesforce-style API. Requires valid Bearer token."""

    def __init__(self, auth: MockAuthServer):
        self.auth = auth
        self.call_count = 0

    async def get_credit_score(self, applicant_id: str, bearer_token: str) -> dict:
        self.call_count += 1
        if not await self.auth.validate(bearer_token):
            raise PermissionError("401 Unauthorized — invalid or expired token")
        seed = sum(ord(c) for c in applicant_id) % 500
        return {
            "applicant_id": applicant_id,
            "score": 350 + seed,
            "tier": "high" if seed >= 350 else "medium" if seed >= 250 else "low",
        }


saas = MockSaaS(auth_server)
print("✓ Mock SaaS resource server ready")

## Step 4 — The TokenCache class

This is the core pattern of Day 3. Fetches a token once, caches it, refreshes 5 minutes (300s) before expiry.

For the lab, tokens last 60s, so we buffer at 10s instead of 300s — so we can watch a refresh happen in real-time.

In [ ]:
class TokenCache:
    BUFFER_SEC = 10   # refresh 10s before expiry (production: 300s)

    def __init__(self, auth_server: MockAuthServer, client_id: str, client_secret: str):
        self.auth = auth_server
        self.client_id = client_id
        self.client_secret = client_secret
        self.value = None
        self.expires_at = 0

    async def get(self) -> str:
        now = time.time()
        if self.value and self.expires_at > now + self.BUFFER_SEC:
            return self.value  # cached, still fresh

        # Refresh
        data = await self.auth.token_endpoint(self.client_id, self.client_secret)
        self.value = data["access_token"]
        self.expires_at = now + data["expires_in"]
        return self.value


cache = TokenCache(auth_server, MockAuthServer.VALID_CLIENT_ID, MockAuthServer.VALID_SECRET)
print("✓ TokenCache initialized")

## Step 5 — Cache behavior: 5 calls, 1 token fetch

In [ ]:
import asyncio


async def demo_cache_hit():
    tokens_seen = set()
    for i in range(5):
        tok = await cache.get()
        tokens_seen.add(tok)
        print(f"  call {i+1}: token = {tok[:20]}...")
    return tokens_seen


tokens = asyncio.run(demo_cache_hit())
print(f"\nUnique tokens across 5 calls: {len(tokens)}")
print(f"Auth server fetch_count:      {auth_server.fetch_count}")
print("\n→ 1 fetch, 5 calls. Cache working.")

## Step 6 — MCP tool wrapping SaaS + token cache

The pattern for Day 3: the agent calls one clean function; auth complexity lives in the wrapper.

In [ ]:
async def mcp_get_credit_score(applicant_id: str) -> dict:
    """MCP-style tool: agent calls this, auth is transparent."""
    token = await cache.get()
    try:
        result = await saas.get_credit_score(applicant_id, token)
    except PermissionError:
        # Token was rejected — force refresh, retry once
        cache.value = None
        cache.expires_at = 0
        token = await cache.get()
        result = await saas.get_credit_score(applicant_id, token)
    return result


result = asyncio.run(mcp_get_credit_score("APP-001"))
print("MCP tool returned:")
print(f"  {result}")
print(f"\nTotal SaaS calls:    {saas.call_count}")
print(f"Total token fetches: {auth_server.fetch_count}")

## Step 7 — Force token expiry, watch refresh happen

Manually expire the token; next call should trigger a fresh fetch.

In [ ]:
async def demo_refresh():
    print(f"Before: fetch_count = {auth_server.fetch_count}")

    # Simulate expiry
    cache.expires_at = time.time() - 1
    print("→ Manually expired the cached token")

    # Next call must refresh
    result = await mcp_get_credit_score("APP-002")
    print(f"After:  fetch_count = {auth_server.fetch_count}")
    print(f"Result: {result}")


asyncio.run(demo_refresh())

## Step 8 — Wire the MCP tool into a LangGraph node

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class LoanState(TypedDict):
    applicant_id: str
    bureau_score: int
    bureau_tier: str


def bureau_node(state: LoanState) -> dict:
    """Sync wrapper around the async MCP tool."""
    result = asyncio.run(mcp_get_credit_score(state["applicant_id"]))
    return {"bureau_score": result["score"], "bureau_tier": result["tier"]}


builder = StateGraph(LoanState)
builder.add_node("bureau", bureau_node)
builder.add_edge(START, "bureau")
builder.add_edge("bureau", END)
graph = builder.compile()

result = graph.invoke({"applicant_id": "APP-999", "bureau_score": 0, "bureau_tier": ""})
print(f"LangGraph result: {result}")

## What you learned

1. **Token cache**: fetch once, reuse until 5 min before expiry
2. **Refresh trigger**: proactive time-based, plus 401-response fallback
3. **Clean agent API**: `mcp_get_credit_score(app_id)` — no auth code in the agent
4. **Retry-once on 401**: token might have been invalidated server-side
5. **LangGraph integration**: sync wrapper around async MCP call

## Production notes

- Real OAuth2: use `httpx.AsyncClient()` against the real token endpoint
- Multi-tenant: one `TokenCache` per tenant (dict keyed by tenant_id)
- Persistent cache: for multi-process, use Redis instead of in-memory dict
- Never log tokens. They're bearer credentials — leak = full access.

## Next

Open `lab2_circuit_breaker.ipynb` for the resilience patterns.
